In [0]:
WITH silver_rows AS (
    SELECT
        'completion_rate' AS table_name,
        CAST(completion_rate_id AS BIGINT) AS row_id,
        year,
        region_code,
        region_name,
        processed_at AS processing_timestamp,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code,
            'indicator', indicator,
            'level_of_education', level_of_education,
            'sex', sex
        )) AS business_key
    FROM pondolens.dev_silver.completion_rate

    UNION ALL

    SELECT
        'grdp_per_capita',
        CAST(grdp_per_capita_id AS BIGINT),
        year, region_code, region_name, processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code,
            'price_basis', price_basis,
            'geographic_level', geographic_level
        ))
    FROM pondolens.dev_silver.grdp_per_capita

    UNION ALL

    SELECT
        'hospital',
        CAST(hospital_id AS BIGINT),
        year, region_code, region_name, processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code,
            'hospital_type', hospital_type
        ))
    FROM pondolens.dev_silver.hospital

    UNION ALL

    SELECT
        'population',
        CAST(population_id AS BIGINT),
        year, region_code, region_name, _processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code,
            'age_group', age_group,
            'sex', sex
        ))
    FROM pondolens.dev_silver.population

    UNION ALL

    SELECT
        'poverty_incidence',
        CAST(poverty_incidence_id AS BIGINT),
        year, region_code, region_name, _processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code,
            'indicator', indicator
        ))
    FROM pondolens.dev_silver.poverty_incidence

    UNION ALL

    SELECT
        'regional_appropriations',
        CAST(gaa_id AS BIGINT),
        year, region_code, region_name, processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'category', category,
            'particular', particular,
            'region_code', region_code
        ))
    FROM pondolens.dev_silver.regional_appropriations

    UNION ALL

    SELECT
        'regional_obligations',
        CAST(obligation_id AS BIGINT),
        year, region_code, region_name, processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'category', category,
            'particular', particular,
            'region_code', region_code
        ))
    FROM pondolens.dev_silver.regional_obligations

    UNION ALL

    SELECT
        'service_access',
        CAST(service_access_id AS BIGINT),
        year, region_code, region_name, processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code,
            'service', service,
            'service_measure', service_measure,
            'breakdown_type', breakdown_type,
            'breakdown_value', breakdown_value
        ))
    FROM pondolens.dev_silver.service_access

    UNION ALL

    SELECT
        'unemployment_rate',
        CAST(unemployment_rate_id AS BIGINT),
        year, region_code, region_name, processed_at,
        TO_JSON(NAMED_STRUCT(
            'year', year,
            'region_code', region_code
        ))
    FROM pondolens.dev_silver.unemployment_rate
),
expected_tables AS (
    SELECT *
    FROM VALUES
        ('completion_rate'),
        ('grdp_per_capita'),
        ('hospital'),
        ('population'),
        ('poverty_incidence'),
        ('regional_appropriations'),
        ('regional_obligations'),
        ('service_access'),
        ('unemployment_rate')
    AS t(table_name)
),
region_reference AS (
    SELECT DISTINCT
        CAST(NULL AS STRING) AS dataset_scope,
        UPPER(TRIM(region_code)) AS region_code,
        UPPER(TRIM(region_name)) AS region_name,
        valid_from_year,
        valid_to_year
    FROM pondolens.dev_silver.shared_region_crosswalk

    UNION ALL

    -- Confirmed dataset-specific Nationwide mappings.
    -- NW remains separate from PHL.
    SELECT
        'regional_appropriations',
        'NW', 'NATIONWIDE', 2021, 2025

    UNION ALL

    SELECT
        'regional_obligations',
        'NW', 'NATIONWIDE', 2021, 2025
),
evaluated AS (
    SELECT
        s.*,
        CASE
            WHEN s.year IS NULL
              OR NULLIF(TRIM(s.region_code), '') IS NULL
              OR NULLIF(TRIM(s.region_name), '') IS NULL
            THEN 1 ELSE 0
        END AS missing_region_fields,

        CASE WHEN EXISTS (
            SELECT 1
            FROM region_reference r
            WHERE (
                r.dataset_scope IS NULL
                OR r.dataset_scope = s.table_name
            )
              AND UPPER(TRIM(s.region_code)) = r.region_code
              AND UPPER(TRIM(s.region_name)) = r.region_name
        ) THEN 1 ELSE 0 END AS known_mapping,

        CASE WHEN EXISTS (
            SELECT 1
            FROM region_reference r
            WHERE (
                r.dataset_scope IS NULL
                OR r.dataset_scope = s.table_name
            )
              AND UPPER(TRIM(s.region_code)) = r.region_code
              AND UPPER(TRIM(s.region_name)) = r.region_name
              AND s.year BETWEEN
                  r.valid_from_year AND r.valid_to_year
        ) THEN 1 ELSE 0 END AS valid_for_year
    FROM silver_rows s
),
row_summary AS (
    SELECT
        table_name,
        COUNT(*) AS total_rows,
        COUNT_IF(
            processing_timestamp IS NULL
        ) AS missing_timestamp_rows,
        COUNT_IF(
            missing_region_fields = 1 OR known_mapping = 0
        ) AS mapping_failed_rows,
        COUNT_IF(
            missing_region_fields = 0 AND known_mapping = 1
        ) AS coverage_evaluated_rows,
        COUNT_IF(
            missing_region_fields = 0
            AND known_mapping = 1
            AND valid_for_year = 0
        ) AS coverage_failed_rows
    FROM evaluated
    GROUP BY table_name
),
key_counts AS (
    SELECT
        table_name,
        row_id,
        COUNT(*) AS rows_per_key
    FROM silver_rows
    GROUP BY table_name, row_id
),
key_summary AS (
    SELECT
        table_name,
        SUM(
            CASE
                WHEN row_id IS NULL OR rows_per_key > 1
                THEN rows_per_key ELSE 0
            END
        ) AS failed_rows
    FROM key_counts
    GROUP BY table_name
),
grain_counts AS (
    SELECT
        table_name,
        business_key,
        COUNT(*) AS rows_per_grain
    FROM silver_rows
    GROUP BY table_name, business_key
),
grain_summary AS (
    SELECT
        table_name,
        SUM(
            CASE WHEN rows_per_grain > 1
                 THEN rows_per_grain ELSE 0 END
        ) AS failed_rows
    FROM grain_counts
    GROUP BY table_name
),
table_summary AS (
    SELECT
        e.table_name,
        COALESCE(r.total_rows, 0) AS total_rows,
        COALESCE(k.failed_rows, 0) AS invalid_id_rows,
        COALESCE(g.failed_rows, 0) AS duplicate_grain_rows,
        COALESCE(r.missing_timestamp_rows, 0) AS missing_timestamp_rows,
        COALESCE(r.mapping_failed_rows, 0) AS mapping_failed_rows,
        COALESCE(r.coverage_evaluated_rows, 0) AS coverage_evaluated_rows,
        COALESCE(r.coverage_failed_rows, 0) AS coverage_failed_rows
    FROM expected_tables e
    LEFT JOIN row_summary r
        ON e.table_name = r.table_name
    LEFT JOIN key_summary k
        ON e.table_name = k.table_name
    LEFT JOIN grain_summary g
        ON e.table_name = g.table_name
),
checks AS (
    SELECT
        table_name,
        'nonempty_table' AS check_name,
        'completeness' AS quality_dimension,
        'critical' AS severity,
        total_rows AS evaluated_rows,
        CAST(0 AS BIGINT) AS failed_rows,
        'Table contains at least one row' AS expected_value
    FROM table_summary

    UNION ALL

    SELECT
        table_name,
        'required_unique_id',
        'uniqueness',
        'critical',
        total_rows,
        invalid_id_rows,
        'ID is nonnull and unique'
    FROM table_summary

    UNION ALL

    SELECT
        table_name,
        'unique_business_grain',
        'uniqueness',
        'critical',
        total_rows,
        duplicate_grain_rows,
        'One row per dataset business-key combination'
    FROM table_summary

    UNION ALL

    SELECT
        table_name,
        'required_processing_timestamp',
        'completeness',
        'critical',
        total_rows,
        missing_timestamp_rows,
        'Every row has a processing timestamp'
    FROM table_summary

    UNION ALL

    SELECT
        table_name,
        'required_recognized_region_mapping',
        'consistency',
        'critical',
        total_rows,
        mapping_failed_rows,
        'Year/code/name present; normalized mapping recognized'
    FROM table_summary

    UNION ALL

    SELECT
        table_name,
        'region_year_coverage_review',
        'coverage',
        'warning',
        coverage_evaluated_rows,
        coverage_failed_rows,
        'Recognized mapping has an explicit validity interval covering year'
    FROM table_summary
)
SELECT
    CONCAT('pondolens.dev_silver.', table_name) AS table_name,
    check_name,
    quality_dimension,
    severity,
    CASE
        WHEN evaluated_rows = 0 AND severity = 'warning'
            THEN 'SKIPPED'
        WHEN evaluated_rows = 0 THEN 'FAIL'
        WHEN failed_rows = 0 THEN 'PASS'
        WHEN severity = 'warning' THEN 'WARN'
        ELSE 'FAIL'
    END AS status,
    evaluated_rows,
    failed_rows,
    expected_value,
    CURRENT_TIMESTAMP() AS checked_at
FROM checks
ORDER BY table_name, check_name;